In [21]:
# ============================================================
# 1. Mount Drive + конфигурация (Colab-версия)
# ============================================================
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
# Опционально: если есть HF_TOKEN, уберёт rate-limit
# os.environ["HF_TOKEN"] = "hf_..."

import re, html, json, time, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from google.colab import drive
drive.mount("/content/drive")

# ----- Пути -----
DRIVE_DIR   = "/content/drive/MyDrive/NewsCategory_results"
DATA_PATH   = "/content/drive/MyDrive/news/News_Category_Dataset_v3.json"
REPORT_DIR  = f"{DRIVE_DIR}/reports"
CACHE_DIR   = f"{DRIVE_DIR}/cache"
MODEL_DIR   = f"{DRIVE_DIR}/distilbert_news_final"
BASELINE_PATH = f"{DRIVE_DIR}/tfidf_svc_baseline.joblib"
# Чекпоинты DistilBERT — локально в Colab (быстрее, чем Drive)
OUTPUT_DIR  = "/content/distilbert_news"

for d in (DRIVE_DIR, REPORT_DIR, CACHE_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

assert os.path.isfile(DATA_PATH), (
    f"Нет датасета по пути {DATA_PATH}. "
    f"Загрузите News_Category_Dataset_v3.json в MyDrive/news/"
)

RANDOM_STATE = 42
TEST_SIZE = 0.20
MAX_FEATURES = 50_000
MIN_SAMPLES = 500

# ----- DistilBERT -----
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN    = 256
BATCH_SIZE = 16
GRAD_ACCUM = 2
EPOCHS     = 3
LR         = 2e-5

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda" and torch.cuda.is_bf16_supported()
USE_FP16 = DEVICE == "cuda" and not USE_BF16

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(
        torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
print("BF16:", USE_BF16, "| FP16:", USE_FP16)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: Tesla T4
VRAM: 15.64 GB
BF16: True | FP16: False


In [22]:
# ============================================================
# 2. Загрузка и подготовка данных
# ============================================================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

try:
    df = pd.read_json(DATA_PATH)
except ValueError:
    df = pd.read_json(DATA_PATH, lines=True)

print("Исходный размер:", df.shape)
df = df.dropna(subset=["category", "headline", "short_description"]).copy()

# Фильтр редких классов
counts = df["category"].value_counts()
valid  = counts[counts >= MIN_SAMPLES].index
df = df[df["category"].isin(valid)].copy()
print(f"Классов после фильтра (>= {MIN_SAMPLES}): {df['category'].nunique()}")

def clean_text(t):
    t = html.unescape(str(t))
    t = re.sub(r"<br\s*/?>", " ", t, flags=re.I)
    t = re.sub(r"<[^>]+>", " ", t)
    t = re.sub(r"\s+", " ", t)
    return t.strip()

df["text"] = df["headline"].astype(str) + " " + df["short_description"].astype(str)
df["text_clean"] = df["text"].apply(clean_text)
df = df[df["text_clean"].str.len() > 0].reset_index(drop=True)

le = LabelEncoder()
df["label"] = le.fit_transform(df["category"])
N_CLASSES = len(le.classes_)

print(f"\nВсего примеров: {len(df):,}")
print(f"Число классов:  {N_CLASSES}")
print("\nРаспределение:")
print(df["category"].value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    df["text_clean"], df["label"],
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["label"],
)
print(f"\nTrain: {len(X_train):,} | Test: {len(X_test):,}")

Исходный размер: (209527, 6)
Классов после фильтра (>= 500): 42

Всего примеров: 209,522
Число классов:  42

Распределение:
category
POLITICS          35601
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6346
FOOD & DRINK       6340
BUSINESS           5991
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2943
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2578
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S

In [23]:
# ============================================================
# 3. Векторайзеры × LogisticRegression + дополнительные конфиги
# ============================================================
import time
from sklearn.feature_extraction.text import (
    CountVectorizer, TfidfVectorizer, HashingVectorizer,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score

vectorizers = {
    "Count (unigram)":        CountVectorizer(stop_words="english"),
    "Count (1-2gram, 50k)":   CountVectorizer(
        ngram_range=(1, 2), stop_words="english", max_features=50_000),
    "TF-IDF (unigram)":       TfidfVectorizer(stop_words="english"),
    "TF-IDF (1-2gram, 50k)":  TfidfVectorizer(
        ngram_range=(1, 2), stop_words="english", max_features=50_000),
    "Hashing (2^18)":         HashingVectorizer(
        n_features=2**18, alternate_sign=False, stop_words="english"),
    # бонусные варианты из «Уровня 1»
    "TF-IDF sublinear":       TfidfVectorizer(
        ngram_range=(1, 2), sublinear_tf=True,
        stop_words="english", max_features=50_000),
    "TF-IDF binary":          TfidfVectorizer(
        ngram_range=(1, 2), binary=True,
        stop_words="english", max_features=50_000),
    "TF-IDF no-stopw":        TfidfVectorizer(
        ngram_range=(1, 2), stop_words=None, max_features=50_000),
    "TF-IDF min_df=3":        TfidfVectorizer(
        ngram_range=(1, 2), min_df=3,
        stop_words="english", max_features=50_000),
}

vec_results = []
for name, vec in vectorizers.items():
    t0 = time.time()
    pipe = Pipeline([
        ("vec", vec),
        ("clf", LogisticRegression(max_iter=1000, n_jobs=-1)),
    ])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_test)
    acc = accuracy_score(y_test, yp)
    f1m = f1_score(y_test, yp, average="macro", zero_division=0)
    f1w = f1_score(y_test, yp, average="weighted", zero_division=0)
    dt  = time.time() - t0
    vec_results.append({
        "vectorizer": name, "accuracy": acc,
        "f1_macro": f1m, "f1_weighted": f1w, "seconds": dt,
    })
    print(f"{name:25s}  acc={acc:.4f}  f1_macro={f1m:.4f}  ({dt:.0f}s)")

vec_df = pd.DataFrame(vec_results).sort_values("f1_macro", ascending=False)
vec_df.to_csv(f"{REPORT_DIR}/vectorizer_comparison.csv", index=False)

print("\n" + "=" * 70)
print("СВОДКА ПО ВЕКТОРАЙЗЕРАМ")
print("=" * 70)
print(vec_df.round(4).to_string(index=False))

KeyboardInterrupt: 

In [ ]:
# ============================================================
# 4. Классификаторы на TF-IDF (1-2gram, sublinear, 50k)
# ============================================================
from sklearn.svm import LinearSVC

def make_best_vec():
    return TfidfVectorizer(
        ngram_range=(1, 2), sublinear_tf=True,
        stop_words="english", max_features=MAX_FEATURES,
    )

classifiers = {
    "LogReg":             LogisticRegression(max_iter=1000, n_jobs=-1),
    "LinearSVC (C=1.0)":  LinearSVC(C=1.0),
    "LinearSVC (C=0.5)":  LinearSVC(C=0.5),
    "LinearSVC (C=0.1)":  LinearSVC(C=0.1),
}

clf_results = []
for name, clf in classifiers.items():
    t0 = time.time()
    pipe = Pipeline([("vec", make_best_vec()), ("clf", clf)])
    pipe.fit(X_train, y_train)
    yp = pipe.predict(X_test)
    acc = accuracy_score(y_test, yp)
    f1m = f1_score(y_test, yp, average="macro", zero_division=0)
    dt  = time.time() - t0
    clf_results.append({"clf": name, "accuracy": acc,
                        "f1_macro": f1m, "seconds": dt})
    print(f"{name:20s}  acc={acc:.4f}  f1_macro={f1m:.4f}  ({dt:.0f}s)")

clf_df = pd.DataFrame(clf_results).sort_values("f1_macro", ascending=False)
clf_df.to_csv(f"{REPORT_DIR}/classifier_comparison.csv", index=False)
print("\n" + clf_df.round(4).to_string(index=False))

In [ ]:
# ============================================================
# 5. Анализ ошибок лучшей пайплайн + сохранение baseline на Drive
# ============================================================
import joblib

best_pipe = Pipeline([
    ("vec", TfidfVectorizer(
        ngram_range=(1, 2), sublinear_tf=True,
        stop_words="english", max_features=MAX_FEATURES,
    )),
    ("clf", LinearSVC(C=0.5)),
])
best_pipe.fit(X_train, y_train)
joblib.dump(best_pipe, BASELINE_PATH)
print(f"Baseline сохранён: {BASELINE_PATH}")

# Для анализа ошибок нужны вероятности — берём LogReg,
# обученный на том же векторайзере (SVC не даёт proba без Platt scaling).
proba_pipe = Pipeline([
    ("vec", TfidfVectorizer(
        ngram_range=(1, 2), sublinear_tf=True,
        stop_words="english", max_features=MAX_FEATURES,
    )),
    ("clf", LogisticRegression(max_iter=1000, n_jobs=-1)),
])
proba_pipe.fit(X_train, y_train)

y_pred_baseline = best_pipe.predict(X_test)
y_proba         = proba_pipe.predict_proba(X_test)
proba_pred      = y_proba.max(axis=1)

errors = pd.DataFrame({
    "text":       X_test.values,
    "true":       y_test.values,
    "pred":       y_pred_baseline,
    "true_cat":   le.inverse_transform(y_test.values),
    "pred_cat":   le.inverse_transform(y_pred_baseline),
    "proba_pred": proba_pred,
})
n_err = int((errors["true"] != errors["pred"]).sum())
print(f"\nОшибок: {n_err:,} из {len(errors):,} "
      f"({n_err/len(errors)*100:.2f}%)")

errors_only = errors[errors["true"] != errors["pred"]].copy()
errors.to_csv(f"{REPORT_DIR}/test_predictions_baseline.csv", index=False)

# Топ-5 самых уверенных ошибок
print("\n" + "=" * 70)
print("Самые уверенные ошибки baseline")
print("=" * 70)
for _, row in errors_only.nlargest(5, "proba_pred").iterrows():
    print(f"\n[true={row['true_cat']} | pred={row['pred_cat']} "
          f"| conf={row['proba_pred']:.2f}]")
    print(row["text"][:300])
    print("-" * 70)

# Топ путаемых пар классов
confused = (errors_only
            .groupby(["true_cat", "pred_cat"])
            .size().reset_index(name="count")
            .sort_values("count", ascending=False).head(10))
print("\nТоп-10 путаемых пар:")
print(confused.to_string(index=False))

In [ ]:
# ============================================================
# 6. GloVe mean-pooling (baseline «плотных» методов)
# ============================================================
import gensim.downloader as api
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

glove = api.load("glove-wiki-gigaword-100")
print(f"GloVe: {len(glove.key_to_index):,} слов, dim={glove.vector_size}")

def text_to_mean_vec(text, model, dim=100):
    words = re.findall(r"[a-z]+", text.lower())
    vecs  = [model[w] for w in words if w in model]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

def vectorize_corpus(texts, model, dim=100):
    return np.vstack([text_to_mean_vec(t, model, dim) for t in tqdm(texts)])

X_tr_glove = vectorize_corpus(X_train.values, glove)
X_te_glove = vectorize_corpus(X_test.values,  glove)

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, n_jobs=-1)),
])
pipe.fit(X_tr_glove, y_train)
yp = pipe.predict(X_te_glove)
glove_acc = accuracy_score(y_test, yp)
glove_f1  = f1_score(y_test, yp, average="macro", zero_division=0)
print(f"\nGloVe mean + LogReg:  acc={glove_acc:.4f}  f1_macro={glove_f1:.4f}")

In [ ]:
# ============================================================
# 7. W2V-SG / W2V-CBOW / FastText — обучение и кэш на Drive
# ============================================================
from gensim.models import Word2Vec, FastText
from gensim.models.word2vec import REAL

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return [t for t in text.split() if len(t) > 1]

print("Токенизация...")
train_tokens = [tokenize(t) for t in tqdm(X_train.values)]
test_tokens  = [tokenize(t) for t in tqdm(X_test.values)]

# IDF-веса для взвешенного усреднения
tfidf_aux = TfidfVectorizer(min_df=2)
tfidf_aux.fit(X_train)
idf_weights = dict(zip(tfidf_aux.get_feature_names_out(), tfidf_aux.idf_))

EMB_DIM = 100
COMMON = dict(vector_size=EMB_DIM, window=5, min_count=5,
              workers=4, epochs=10, seed=RANDOM_STATE)

def load_or_train(cls, sg, name):
    path = f"{CACHE_DIR}/{name}.model"
    if os.path.exists(path):
        print(f"Загружаем {name} из кэша")
        return cls.load(path)
    print(f"Обучаем {name}...")
    m = cls(sg=sg, **COMMON)
    m.build_vocab(train_tokens)
    m.wv.vectors_lockf = np.ones(len(m.wv), dtype=REAL)
    m.train(train_tokens, total_examples=len(train_tokens),
            epochs=COMMON["epochs"])
    m.save(path)
    return m

w2v_sg   = load_or_train(Word2Vec, sg=1, name="w2v_sg")
w2v_cbow = load_or_train(Word2Vec, sg=0, name="w2v_cbow")
ft       = load_or_train(FastText, sg=1, name="fasttext")

def mean_vec(tokens, wv):
    vs = [wv[w] for w in tokens if w in wv]
    return np.mean(vs, axis=0) if vs else np.zeros(EMB_DIM)

def idf_vec(tokens, wv):
    vs, ws = [], []
    for w in tokens:
        if w in wv:
            vs.append(wv[w]); ws.append(idf_weights.get(w, 1.0))
    if not vs: return np.zeros(EMB_DIM)
    vs = np.array(vs); ws = np.array(ws).reshape(-1, 1)
    return (vs * ws).sum(axis=0) / (ws.sum() + 1e-9)

def vectorize(tokens_list, wv, mode="mean"):
    fn = mean_vec if mode == "mean" else idf_vec
    return np.vstack([fn(t, wv) for t in tqdm(tokens_list, leave=False)])

def evaluate_emb(Xtr, Xte, label):
    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, n_jobs=-1)),
    ])
    clf.fit(Xtr, y_train)
    yp = clf.predict(Xte)
    acc = accuracy_score(y_test, yp)
    f1  = f1_score(y_test, yp, average="macro", zero_division=0)
    print(f"{label:30s}  acc={acc:.4f}  f1_macro={f1:.4f}")
    return acc, f1

emb_results = {}
for mname, wv in [("W2V-SG", w2v_sg.wv),
                  ("W2V-CBOW", w2v_cbow.wv),
                  ("FastText", ft.wv)]:
    for mode in ("mean", "idf"):
        Xtr = vectorize(train_tokens, wv, mode)
        Xte = vectorize(test_tokens,  wv, mode)
        emb_results[f"{mname} ({mode})"] = evaluate_emb(Xtr, Xte, f"{mname} ({mode})")

In [ ]:
# ============================================================
# 8. Sentence-Transformers MiniLM (с кэшем на Drive)
# ============================================================
from sentence_transformers import SentenceTransformer

MINILM_CACHE = f"{CACHE_DIR}/minilm_emb.npz"

if os.path.exists(MINILM_CACHE):
    print(f"Загружаем MiniLM-эмбеддинги из {MINILM_CACHE}")
    data = np.load(MINILM_CACHE)
    X_tr_mini = data["X_tr_mini"]
    X_te_mini = data["X_te_mini"]
else:
    print("Считаем MiniLM-эмбеддинги (10–15 мин)...")
    st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    X_tr_mini = st.encode(X_train.tolist(), batch_size=64,
                          show_progress_bar=True).astype(np.float32)
    X_te_mini = st.encode(X_test.tolist(),  batch_size=64,
                          show_progress_bar=True).astype(np.float32)
    np.savez_compressed(MINILM_CACHE,
                        X_tr_mini=X_tr_mini, X_te_mini=X_te_mini)
    print(f"Сохранено: {MINILM_CACHE}")

print(f"MiniLM: train {X_tr_mini.shape}, test {X_te_mini.shape}")

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf",    LogisticRegression(max_iter=1000, n_jobs=-1)),
])
pipe.fit(X_tr_mini, y_train)
yp = pipe.predict(X_te_mini)
mini_acc = accuracy_score(y_test, yp)
mini_f1  = f1_score(y_test, yp, average="macro", zero_division=0)
print(f"\nMiniLM + LogReg:  acc={mini_acc:.4f}  f1_macro={mini_f1:.4f}")

In [ ]:
# ============================================================
# 9. Гибрид TF-IDF (SVD) + MiniLM
# ============================================================
from sklearn.decomposition import TruncatedSVD
from scipy.sparse import csr_matrix

# 1. TF-IDF
tfidf = TfidfVectorizer(
    ngram_range=(1, 2), sublinear_tf=True,
    stop_words="english", max_features=MAX_FEATURES,
)
X_tr_sp = tfidf.fit_transform(X_train)
X_te_sp = tfidf.transform(X_test)
print(f"TF-IDF: {X_tr_sp.shape}, nnz={X_tr_sp.nnz:,}")

# 2. MiniLM + скейлер, вес плотного блока = 0.3
scaler = StandardScaler().fit(X_tr_mini)
X_tr_d = (scaler.transform(X_tr_mini) * 0.3).astype(np.float32)
X_te_d = (scaler.transform(X_te_mini) * 0.3).astype(np.float32)

# 3. SVD: 50k → 300 компонент
svd = TruncatedSVD(n_components=300, random_state=RANDOM_STATE)
X_tr_svd = svd.fit_transform(X_tr_sp).astype(np.float32)
X_te_svd = svd.transform(X_te_sp).astype(np.float32)
print(f"SVD объясняет {svd.explained_variance_ratio_.sum():.3f} дисперсии")

# 4. Конкатенация (плотное + плотное — простая numpy-операция)
X_tr_hyb = np.hstack([X_tr_svd, X_tr_d])
X_te_hyb = np.hstack([X_te_svd, X_te_d])
print(f"Hybrid: {X_tr_hyb.shape}  ({X_tr_hyb.nbytes/1e6:.0f} MB)")

clf = LinearSVC(C=0.5, max_iter=2000).fit(X_tr_hyb, y_train)
yp  = clf.predict(X_te_hyb)
hyb_acc = accuracy_score(y_test, yp)
hyb_f1  = f1_score(y_test, yp, average="macro", zero_division=0)
print(f"\nHybrid (SVD+MiniLM) + LinearSVC:  acc={hyb_acc:.4f}  f1_macro={hyb_f1:.4f}")

In [ ]:
# ============================================================
# 10. Сводная таблица (все подходы)
# ============================================================
summary = []

# Векторайзеры + LogReg
for _, r in vec_df.iterrows():
    summary.append({"method": r["vectorizer"],
                    "accuracy": r["accuracy"], "f1_macro": r["f1_macro"]})

# Классификаторы на лучшем векторайзере
for _, r in clf_df.iterrows():
    summary.append({"method": f"TF-IDF sublinear + {r['clf']}",
                    "accuracy": r["accuracy"], "f1_macro": r["f1_macro"]})

# Эмбеддинги
summary.append({"method": "GloVe mean + LogReg",
                "accuracy": glove_acc, "f1_macro": glove_f1})
for name, (acc, f1) in emb_results.items():
    summary.append({"method": f"{name} + LogReg",
                    "accuracy": acc, "f1_macro": f1})

# MiniLM и гибрид
summary.append({"method": "MiniLM + LogReg",
                "accuracy": mini_acc, "f1_macro": mini_f1})
summary.append({"method": "Hybrid SVD+MiniLM + LinearSVC",
                "accuracy": hyb_acc, "f1_macro": hyb_f1})

summary_df = (pd.DataFrame(summary)
              .sort_values("f1_macro", ascending=False)
              .reset_index(drop=True))
summary_df.to_csv(f"{REPORT_DIR}/baseline_summary.csv", index=False)

print("=" * 75)
print("ИТОГОВАЯ СВОДКА BASELINE")
print("=" * 75)
print(summary_df.round(4).to_string(index=False))

# Bar-chart для отчёта
fig, ax = plt.subplots(figsize=(11, 0.4 * len(summary_df) + 1.5))
ax.barh(summary_df["method"], summary_df["f1_macro"])
ax.invert_yaxis()
ax.set_xlabel("F1 macro")
ax.set_title("Классические baseline'ы на News Category Dataset")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()
fig.savefig(f"{REPORT_DIR}/baseline_summary.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 11. DistilBERT — подготовка
# ============================================================
from torch.utils.data import Dataset, Subset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    Trainer, TrainingArguments, DataCollatorWithPadding,
    TrainerCallback,
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_texts(texts, tok, max_len):
    return tok(list(texts), add_special_tokens=True, truncation=True,
               max_length=max_len, padding=False, return_attention_mask=True)

train_enc = tokenize_texts(X_train.tolist(), tokenizer, MAX_LEN)
val_enc   = tokenize_texts(X_test.tolist(),  tokenizer, MAX_LEN)  # используем test как eval
# Для полноты сделаем отдельный val из train
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train, y_train, test_size=0.125,
    random_state=RANDOM_STATE, stratify=y_train,
)
train_enc = tokenize_texts(X_tr.tolist(),  tokenizer, MAX_LEN)
val_enc   = tokenize_texts(X_val.tolist(), tokenizer, MAX_LEN)
test_enc  = tokenize_texts(X_test.tolist(), tokenizer, MAX_LEN)

class NewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx], dtype=torch.long)
                for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

train_ds = NewsDataset(train_enc, y_tr.tolist())
val_ds   = NewsDataset(val_enc,   y_val.tolist())
test_ds  = NewsDataset(test_enc,  y_test.tolist())

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8 if DEVICE == "cuda" else None,
)

print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")

In [ ]:
# ============================================================
# 12. Метрики и callback
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred.predictions, eval_pred.label_ids
    if isinstance(logits, tuple):
        logits = logits[0]
    preds = np.argmax(logits, axis=-1)
    p, r, f, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f, "precision_macro": p, "recall_macro": r}

class ProgressCallback(TrainerCallback):
    def __init__(self, desc="DistilBERT"):
        self.desc = desc; self.bar = None
        self.start_time = None; self.last_step = 0
    def on_train_begin(self, args, state, control, **kw):
        from tqdm.auto import tqdm
        self.start_time = time.time()
        self.last_step = state.global_step
        total = state.max_steps if state.max_steps and state.max_steps > 0 else None
        self.bar = tqdm(total=total, initial=state.global_step,
                        desc=self.desc, unit="step", dynamic_ncols=True)
    def on_step_end(self, args, state, control, **kw):
        if self.bar is None: return
        d = state.global_step - self.last_step
        if d > 0:
            self.bar.update(d); self.last_step = state.global_step
        elapsed = time.time() - self.start_time
        loss = next((e["loss"] for e in reversed(state.log_history)
                     if "loss" in e), None)
        eta = (elapsed / state.global_step *
               max(0, state.max_steps - state.global_step)) \
              if state.max_steps and state.global_step > 0 else None
        pf = {}
        if loss is not None: pf["loss"] = f"{loss:.4f}"
        if eta  is not None: pf["ETA"]  = f"{eta/60:.1f} min"
        self.bar.set_postfix(pf)
    def on_train_end(self, args, state, control, **kw):
        if self.bar: self.bar.close(); self.bar = None

print("Метрики и callback готовы.")

In [ ]:
# ============================================================
# 13. Smoke-test
# ============================================================
SMOKE_SIZE, SMOKE_VAL = 256, 128
smoke_train = Subset(train_ds, range(min(SMOKE_SIZE, len(train_ds))))
smoke_val   = Subset(val_ds,   range(min(SMOKE_VAL,   len(val_ds))))

smoke_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=N_CLASSES,
    id2label={i: c for i, c in enumerate(le.classes_)},
    label2id={c: i for i, c in enumerate(le.classes_)},
)

smoke_args = TrainingArguments(
    output_dir="./smoke_check", num_train_epochs=1,
    per_device_train_batch_size=4, per_device_eval_batch_size=8,
    gradient_accumulation_steps=1, learning_rate=1e-5,
    warmup_steps=0, weight_decay=0.01, fp16=False,
    eval_strategy="epoch", save_strategy="no",
    logging_strategy="steps", logging_steps=10,
    report_to="none", disable_tqdm=True,
    dataloader_num_workers=0,
    seed=RANDOM_STATE, data_seed=RANDOM_STATE,
)

smoke_trainer = Trainer(
    model=smoke_model, args=smoke_args,
    train_dataset=smoke_train, eval_dataset=smoke_val,
    data_collator=data_collator, compute_metrics=compute_metrics,
)

before = smoke_trainer.evaluate(eval_dataset=smoke_train, metric_key_prefix="before")
print("Loss до:", before["before_loss"])
smoke_trainer.train()
after  = smoke_trainer.evaluate(eval_dataset=smoke_train, metric_key_prefix="after")
print("Loss после:", after["after_loss"])
vm = smoke_trainer.evaluate(eval_dataset=smoke_val, metric_key_prefix="smoke_val")
print("Smoke val:", vm)

In [ ]:
# ============================================================
# 14. Основная модель DistilBERT
# ============================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=N_CLASSES,
    id2label={i: c for i, c in enumerate(le.classes_)},
    label2id={c: i for i, c in enumerate(le.classes_)},
)
print(f"Параметров: {model.num_parameters():,}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=0, weight_decay=0.01,
    fp16=USE_FP16, bf16=USE_BF16,
    eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=1, load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True,
    logging_strategy="steps", logging_steps=50,
    report_to="none", disable_tqdm=True,
    seed=RANDOM_STATE, data_seed=RANDOM_STATE,
    dataloader_num_workers=2,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[ProgressCallback("DistilBERT training")],
)
print("Trainer готов. Обучение ещё не запущено.")

In [ ]:
# ============================================================
# 15. Обучение DistilBERT
# ============================================================
start_time = time.time()
train_result = trainer.train()
training_seconds = time.time() - start_time

print(f"\nTraining time: {training_seconds/60:.2f} min")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best val metric: {trainer.state.best_metric}")

In [ ]:
# ============================================================
# 16. Финальная оценка на тесте
# ============================================================
pred_output = trainer.predict(test_ds, metric_key_prefix="test")
print(pred_output.metrics)

bert_logits = pred_output.predictions
if isinstance(bert_logits, tuple):
    bert_logits = bert_logits[0]

y_pred_bert = np.argmax(bert_logits, axis=-1)
bert_probs  = torch.softmax(
    torch.tensor(bert_logits, dtype=torch.float32), dim=-1).numpy()

bert_acc = accuracy_score(y_test, y_pred_bert)
bert_f1  = f1_score(y_test, y_pred_bert, average="macro", zero_division=0)
print(f"\nDistilBERT test:  acc={bert_acc:.4f}  f1_macro={bert_f1:.4f}")

print("\nClassification report — DistilBERT:")
print(classification_report(
    y_test, y_pred_bert,
    target_names=le.classes_, zero_division=0,
))

In [ ]:
# ============================================================
# 17. Визуализация
# ============================================================
history = trainer.state.log_history
train_logs = [x for x in history if "loss" in x and "eval_accuracy" not in x]
val_logs   = [x for x in history if "eval_accuracy" in x]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if train_logs:
    axes[0].plot([x["step"] for x in train_logs],
                 [x["loss"] for x in train_logs], linewidth=1.5)
axes[0].set_xlabel("Step"); axes[0].set_ylabel("Training loss")
axes[0].set_title("Training loss"); axes[0].grid(alpha=0.3)

if val_logs:
    steps = [x["step"] for x in val_logs]
    accs  = [x["eval_accuracy"] for x in val_logs]
    f1s   = [x["eval_f1_macro"] for x in val_logs]
    axes[1].plot(steps, accs, marker="o", label="accuracy")
    axes[1].plot(steps, f1s,  marker="s", label="f1_macro")
axes[1].set_xlabel("Step"); axes[1].set_ylabel("Score")
axes[1].set_title("Validation metrics"); axes[1].grid(alpha=0.3)
axes[1].legend()

fig.tight_layout()
fig.savefig(f"{REPORT_DIR}/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()

# Матрица ошибок (большая, сохраняем в файл)
cm = confusion_matrix(y_test, y_pred_bert)
fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(cm, annot=False, cmap="Blues",
            xticklabels=le.classes_, yticklabels=le.classes_,
            cbar=True, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("DistilBERT — Confusion matrix")
plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)
fig.tight_layout()
fig.savefig(f"{REPORT_DIR}/confusion_matrix_distilbert.png",
            dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 18. Финальное сравнение
# ============================================================
baseline_row = summary_df.iloc[0]  # лучший классический

comparison = pd.DataFrame({
    "Method": [f"Baseline: {baseline_row['method']}", "DistilBERT (fine-tuned)"],
    "Accuracy": [baseline_row["accuracy"], bert_acc],
    "F1 macro": [baseline_row["f1_macro"], bert_f1],
})
print("=" * 70)
print("BASELINE vs DISTILBERT")
print("=" * 70)
print(comparison.round(4).to_string(index=False))
print(f"\nΔ F1 macro: {(bert_f1 - baseline_row['f1_macro'])*100:+.2f} п.п.")
print(f"Δ Accuracy: {(bert_acc - baseline_row['accuracy'])*100:+.2f} п.п.")

comparison.to_csv(f"{REPORT_DIR}/final_comparison.csv", index=False)

# Сохраняем модель и токенизатор на Drive
os.makedirs(MODEL_DIR, exist_ok=True)
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)

# JSON-отчёт
report = {
    "model": MODEL_NAME, "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM,
    "epochs": EPOCHS, "lr": LR,
    "num_classes": N_CLASSES,
    "classes": le.classes_.tolist(),
    "train_size": len(train_ds), "val_size": len(val_ds),
    "test_size": len(test_ds),
    "training_seconds": training_seconds,
    "baseline": {
        "method": baseline_row["method"],
        "accuracy": float(baseline_row["accuracy"]),
        "f1_macro": float(baseline_row["f1_macro"]),
    },
    "distilbert": {"accuracy": float(bert_acc), "f1_macro": float(bert_f1)},
}
with open(f"{REPORT_DIR}/experiment_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(f"\nМодель:  {MODEL_DIR}/")
print(f"Отчёты:  {REPORT_DIR}/")